# GOOGLE TIMESFM - Binary Quant X V16 Supreme
**Voto de Minerva com GPU do Google Colab (T4 gratuita)**

---
1. Baixa TimesFM real (500M params) na GPU T4
2. Carrega ultimas 512 candles do IQ Option
3. Roda previsao para as proximas 4 velas
4. Exporta JSON que o GitHub Actions consome

**Run:** Runtime - Run all (~3 min)
**Resultado:** timesfm_previsao.json baixado automaticamente

In [ ]:
!pip install -q git+https://github.com/google-research/timesfm.git
!pip install -q numpy pandas torch

import os, json, sys
import numpy as np
import pandas as pd
import torch
from datetime import datetime
from google.colab import files

print(f"GPU disponivel: {torch.cuda.is_available()}")if torch.cuda.is_available():
    print(f'GPU: {torch.cuda.get_device_name(0)}')    print(f'VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB')

In [ ]:
%%time
import timesfm
print('Baixando Google TimesFM 2.0-500m...')tfm = timesfm.TimesFm(    hparams=timesfm.TimesFmHparams(backend='torch', num_layers=50, context_len=512, horizon_len=4),    checkpoint=timesfm.TimesFmCheckpoint(huggingface_repo_id='google/timesfm-2.0-500m-jax'),)if torch.cuda.is_available():    tfm._model = tfm._model.cuda()    print('Modelo movido para GPU T4')n_params = sum(p.numel() for p in tfm._model.parameters())print(f'Modelo carregado! {n_params:,} parametros')

## ENTRADA DE DADOS

Upload CSV (close,open,high,low,volume) ou use dados simulados.

In [ ]:
# Upload manual:# uploaded = files.upload()# csv_name = list(uploaded.keys())[0]# df = pd.read_csv(csv_name)# Dados simulados EURUSD:np.random.seed(42)n = 512trend = np.linspace(1.0800, 1.0950, n)noise = np.random.normal(0, 0.0005, n)close = trend + noisedf = pd.DataFrame({    'close': close,    'open': close + np.random.normal(0, 0.0003, n),    'high': close + np.abs(np.random.normal(0, 0.0008, n)),    'low': close - np.abs(np.random.normal(0, 0.0008, n)),    'volume': np.random.randint(1000, 100000, n)})print(f'{len(df)} candles')

In [ ]:
%%timeprint('Rodando inferencia do TimesFM na GPU...')input_arr = np.array(df['close'].values, dtype=np.float32).reshape(1, -1)with torch.no_grad():    forecast_mean, forecast_std = tfm.forecast(input_arr)print(f'Previsao (4 velas): {forecast_mean[0].round(5)}')print(f'Confianca (std): {forecast_std[0].round(5)}')

## EXPORT JSON PARA O GHA

In [ ]:
ultimo = df['close'].values[-1]previsto = forecast_mean[0][-1]variacao = (previsto - ultimo) / ultimo * 100if variacao > 0.05: direcao = 'UP'elif variacao < -0.05: direcao = 'DOWN'else: direcao = 'NEUTRAL'conf = min(abs(variacao * 10), 0.99)resultado = {    'timestamp': datetime.now().strftime('%Y-%m-%d %H:%M:%S'),    'modelo': 'google/timesfm-2.0-500m',    'gpu': torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'CPU',    'ultimo_preco': round(float(ultimo), 5),    'previsao_velas': [round(float(v), 5) for v in forecast_mean[0]],    'direcao': direcao,    'confianca': round(conf, 4),    'importancia': 'VOTO_DE_MINERVA'}with open('timesfm_previsao.json', 'w') as f:    json.dump(resultado, f, indent=2)print('='*50)print('RESULTADO TIMESFM V16 SUPREME')print('='*50)print(json.dumps(resultado, indent=2))print('='*50)files.download('timesfm_previsao.json')print('JSON baixado! Suba no repo e o GHA usa como Voto de Minerva.')

## COMO USAR

1. Rode a cada 1-2 horas
2. Baixe o timesfm_previsao.json
3. Suba no repo alinenerin/sniper-v9-iqoption
4. GHA le o JSON como Voto de Minerva

Dica: Automatize com gspread ou webhook.

In [ ]:
# AUTO PUSH PRO GITHUB
import requests, os, base64, json, datetime
from google.colab import userdata

try:
    gh_token = userdata.get('GH_TOKEN')
except:
    gh_token = None

if gh_token:
    print('Token encontrado! Enviando previsao...')
    headers = {'Authorization': f'Bearer {gh_token}', 'Accept': 'application/vnd.github.v3+json'}
    repo_url = 'https://api.github.com/repos/alinenerin/sniper-v9-iqoption'

    r = requests.get(f'{repo_url}/git/refs/heads/main', headers=headers)
    head_sha = r.json()['object']['sha']

    r = requests.get(f'{repo_url}/git/commits/{head_sha}', headers=headers)
    base_tree = r.json()['tree']['sha']

    with open('timesfm_previsao.json', 'rb') as f:
        b64 = base64.b64encode(f.read()).decode()
    r = requests.post(f'{repo_url}/git/blobs', headers=headers, json={'content': b64, 'encoding': 'base64'})
    blob_sha = r.json()['sha']

    r = requests.post(f'{repo_url}/git/trees', headers=headers, json={
        'base_tree': base_tree,
        'tree': [{'path': 'previsao_timesfm.json', 'mode': '100644', 'type': 'blob', 'sha': blob_sha}]
    })
    new_tree = r.json()['sha']

    now = datetime.datetime.now().strftime('%Y-%m-%d %H:%M')
    r = requests.post(f'{repo_url}/git/commits', headers=headers, json={
        'message': f'auto: previsao TimesFM {now}',
        'tree': new_tree, 'parents': [head_sha]
    })
    commit_sha = r.json()['sha']

    r = requests.patch(f'{repo_url}/git/refs/heads/main', headers=headers, json={
        'sha': commit_sha, 'force': False
    })
    print(f'Push feito! Commit: {commit_sha[:12]}')
    print('Seu GHA vai ler o previsao_timesfm.json agora!')
else:
    print('Sem token. Configure GH_TOKEN no Colab (gear > Secrets) ou baixe manualmente.')
    print('files.download("timesfm_previsao.json")')